# 03 - CSV para Delta Lake (MinIO)

Lê os arquivos CSV do bucket `landing-zone` no MinIO e grava cada um como tabela Delta Lake no bucket `bronze`.

**Pré-requisitos:** Notebook `02` executado (CSVs no MinIO).

## 1. Imports e Configuração

In [1]:
import os
import boto3
from botocore.client import Config
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from delta import *

load_dotenv(override=True)

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
LANDING_BUCKET   = os.getenv('MINIO_LANDING_BUCKET')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET')

print(f'MinIO: {MINIO_ENDPOINT}')
print(f'Landing: {LANDING_BUCKET} | Bronze: {BRONZE_BUCKET}')

MinIO: http://localhost:9020
Landing: landing-zone | Bronze: bronze


## 2. Criar SparkSession com Delta Lake e MinIO

In [2]:
spark = (
    SparkSession.builder
    .appName('CSV_to_Delta')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    # MinIO / S3A
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)
print('SparkSession criada com sucesso!')
spark

26/05/04 19:42:34 WARN Utils: Your hostname, DESKTOP-C71TG2N resolves to a loopback address: 127.0.1.1; using 172.22.160.68 instead (on interface eth0)
26/05/04 19:42:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/ian/spark-delta-minio-sqlserver/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ian/.ivy2/cache
The jars for the packages stored in: /home/ian/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d94efb59-737f-44eb-aaa6-e199e0b4c3cb;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 576ms :: artifacts dl 29ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.hadoop#had

SparkSession criada com sucesso!


## 3. Criar Bucket Bronze no MinIO

In [3]:
s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

try:
    s3_client.head_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket [{BRONZE_BUCKET}] ja existe')
except:
    s3_client.create_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket [{BRONZE_BUCKET}] criado!')

print('Buckets:', [b['Name'] for b in s3_client.list_buckets()['Buckets']])

Bucket [bronze] criado!
Buckets: ['bronze', 'landing-zone']


## 4. Listar CSVs Disponíveis no Landing Zone

In [4]:
response = s3_client.list_objects_v2(Bucket=LANDING_BUCKET)
csv_files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.csv')]

print(f'{len(csv_files)} arquivos CSV encontrados no bucket [{LANDING_BUCKET}]:')
for f in csv_files:
    print(f'  - {f}')

5 arquivos CSV encontrados no bucket [landing-zone]:
  - categorias.csv
  - clientes.csv
  - itens_venda.csv
  - produtos.csv
  - vendas.csv


## 5. Ler CSVs e Gravar como Delta Lake

In [5]:
from delta.tables import DeltaTable

print(f'Convertendo {len(csv_files)} CSVs para Delta Lake...\n')

for csv_file in csv_files:
    tabela = csv_file.replace('.csv', '')
    csv_path = f's3a://{LANDING_BUCKET}/{csv_file}'
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'
    
    # Ler CSV com inferência de schema
    df = spark.read \
        .option('header', 'true') \
        .option('inferSchema', 'true') \
        .csv(csv_path)
    
    # Gravar como Delta Lake
    df.write \
        .format('delta') \
        .mode('overwrite') \
        .save(delta_path)
    
    print(f'  {tabela}: {df.count()} registros | {len(df.columns)} colunas -> {delta_path}')

print(f'\nConversao concluida! {len(csv_files)} tabelas Delta criadas no bucket [{BRONZE_BUCKET}].')

Convertendo 5 CSVs para Delta Lake...



26/05/04 19:42:52 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/05/04 19:42:57 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

  categorias: 5 registros | 3 colunas -> s3a://bronze/categorias
  clientes: 10 registros | 4 colunas -> s3a://bronze/clientes
  itens_venda: 13 registros | 5 colunas -> s3a://bronze/itens_venda
  produtos: 10 registros | 5 colunas -> s3a://bronze/produtos


  vendas: 10 registros | 4 colunas -> s3a://bronze/vendas

Conversao concluida! 5 tabelas Delta criadas no bucket [bronze].


## 6. Validação - Ler Tabelas Delta

In [6]:
print('Validando tabelas Delta Lake...\n')

for csv_file in csv_files:
    tabela = csv_file.replace('.csv', '')
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'
    
    # Verificar se é Delta
    is_delta = DeltaTable.isDeltaTable(spark, delta_path)
    df_delta = spark.read.format('delta').load(delta_path)
    
    print(f'  {tabela}: Delta={is_delta} | {df_delta.count()} registros | Colunas: {df_delta.columns}')

Validando tabelas Delta Lake...



26/05/04 19:43:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

  categorias: Delta=True | 5 registros | Colunas: ['id_categoria', 'nome_categoria', 'descricao']


  clientes: Delta=True | 10 registros | Colunas: ['id_cliente', 'nome', 'estado', 'status_conta']


  itens_venda: Delta=True | 13 registros | Colunas: ['id_item', 'id_venda', 'id_produto', 'quantidade', 'preco_unitario']


  produtos: Delta=True | 10 registros | Colunas: ['id_produto', 'nome', 'id_categoria', 'preco', 'estoque']


[Stage 66:==================================================>     (45 + 5) / 50]

  vendas: Delta=True | 10 registros | Colunas: ['id_venda', 'id_cliente', 'data_venda', 'valor_total']


In [7]:
# Amostra: exibir primeiros registros de algumas tabelas
for tabela in ['categorias', 'produtos', 'vendas']:
    print(f'\n--- {tabela.upper()} ---')
    spark.read.format('delta').load(f's3a://{BRONZE_BUCKET}/{tabela}').show(5)


--- CATEGORIAS ---


+------------+--------------+--------------------+
|id_categoria|nome_categoria|           descricao|
+------------+--------------+--------------------+
|           1|   Eletrônicos|Smartphones, note...|
|           2|     Vestuário|Roupas masculinas...|
|           3|        Livros|Ficção, técnicos ...|
|           4|          Casa|Móveis e itens de...|
|           5|      Esportes|Artigos esportivo...|
+------------+--------------+--------------------+


--- PRODUTOS ---


+----------+------------------+------------+------+-------+
|id_produto|              nome|id_categoria| preco|estoque|
+----------+------------------+------------+------+-------+
|         1|    Smartphone XYZ|           1|1500.0|     50|
|         2|      Notebook Pro|           1|3500.0|     30|
|         3|   Camiseta Básica|           2|  50.0|    100|
|         4|       Calça Jeans|           2| 120.0|     80|
|         5|O Senhor dos Anéis|           3|  60.0|     40|
+----------+------------------+------------+------+-------+
only showing top 5 rows


--- VENDAS ---


[Stage 89:==========================================>             (38 + 8) / 50]

+--------+----------+----------+-----------+
|id_venda|id_cliente|data_venda|valor_total|
+--------+----------+----------+-----------+
|       1|         1|2026-05-01|     1500.0|
|       2|         2|2026-05-02|     3500.0|
|       3|         4|2026-05-02|      170.0|
|       4|         5|2026-05-03|      145.0|
|       5|         7|2026-05-03|     1200.0|
+--------+----------+----------+-----------+
only showing top 5 rows



In [8]:
spark.stop()
print('SparkSession finalizada.')

SparkSession finalizada.
